# BellaBox Product Vision CNN

هذا الدفتر يبني Dataset حقيقيًا من صور منتجات متجر بيلابوكس العامة، ثم يدرب CNN لتصنيف فئة المنتج. لا يستخدم Dataset تعليميًا جاهزًا. شغّل الخلايا بالترتيب، وراجع `manifest.csv` و`dataset_summary.json` قبل اعتماد النتائج.

**مهم:** فعّل GPU من `Runtime > Change runtime type > T4 GPU`، واربط Google Drive حتى تبقى Checkpoints بعد انتهاء جلسة Colab.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/newbellabox')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mohammedalhmed/newbellabox.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'ml/requirements-colab.txt')], check=True)
print('Repository ready:', REPO_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/BellaBox_Product_CNN')
DATA_DIR = WORK_DIR / 'dataset'
OUTPUT_DIR = WORK_DIR / 'outputs'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent work directory:', WORK_DIR)

## 1) بناء Dataset من Sitemap بيلابوكس

يحتوي Sitemap الحالي على روابط منتجات وصور CDN. الأداة تحفظ صورة/صور كل منتج في مجلد الفئة وتكتب كل قرار في `manifest.csv`. إذا كانت بعض التسميات غير مناسبة، عدّل عمود `label` قبل الانتقال للتدريب.

In [ ]:
import subprocess
cmd = [
    sys.executable, str(REPO_DIR / 'ml/build_dataset.py'),
    '--output-dir', str(DATA_DIR),
    '--min-images-per-class', '10',
    '--max-images-per-product', '3',
    '--download',
]
subprocess.run(cmd, check=True)

In [ ]:
summary = json.loads((DATA_DIR / 'dataset_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Review this summary and manifest.csv before training.')

## 2) تدريب CNN مع Checkpoints

التدريب ينفذ Grouped Split حسب `product_id`، ويستخدم EfficientNetB0، Augmentation، Class Weights، Early Stopping، ReduceLROnPlateau، و`BackupAndRestore`. يمكن إعادة تشغيل الخلية بعد انقطاع Colab؛ سيقرأ ملفات الاستئناف الموجودة في Drive.

In [ ]:
train_cmd = [
    sys.executable, str(REPO_DIR / 'ml/train.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '15',
    '--batch-size', '32',
    '--resume',
]
subprocess.run(train_cmd, check=True)

In [ ]:
metrics_path = OUTPUT_DIR / 'metrics.json'
print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
print('Saved files:', sorted(p.name for p in OUTPUT_DIR.iterdir()))

## 3) اختبار النموذج على صورة جديدة

ارفع صورة منتج من خارج Dataset إن أمكن، ثم راقب Top-3. إذا كانت الثقة منخفضة أو الصورة مختلفة جدًا عن صور التدريب، تُحال النتيجة للمراجعة البشرية.

In [ ]:
from google.colab import files
uploaded = files.upload()
test_image = next(iter(uploaded))
predict_cmd = [
    sys.executable, str(REPO_DIR / 'ml/predict.py'),
    '--model', str(OUTPUT_DIR / 'final_model.keras'),
    '--labels', str(OUTPUT_DIR / 'labels.json'),
    '--image', test_image,
    '--top-k', '3',
]
subprocess.run(predict_cmd, check=True)